## Imports

In [1]:
import warnings
warnings.filterwarnings('ignore', message='.*PyTorch API of nested tensors.*')
import math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR   # LR decay scheduler
from torch.amp import autocast, GradScaler           # Mixed precision (AMP)
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from torchvision.transforms import functional as VF
from collections import defaultdict, Counter
from typing import List, Tuple
import re, os
from torch.optim.swa_utils import AveragedModel       # EMA of model weights
from torch.utils.tensorboard import SummaryWriter      # TensorBoard logging


## Hyperparameters

In [2]:
# ── Core dimensions ──────────────────────────────────────────────────────────
EMBD_DIM    = 256        # Embedding dimension for both encoders
ATTEN_HEADS = 4          # Attention heads in transformer
NUM_TX_LAYERS = 6        # Transformer depth
DROPOUT     = 0.1        # Dropout for regularisation

# ── Training settings ─────────────────────────────────────────────────────────
batch_size  = 256        # Large batch = more negatives for contrastive learning
epochs      = 50
LR          = 3e-4       # AdamW learning rate
WEIGHT_DECAY = 1e-4      # L2-style regularisation for AdamW
WARMUP_EPOCHS = 3        # LR warmup before cosine decay
EARLY_STOP_PATIENCE = 8  # stop if val loss does not improve
LABEL_SMOOTH = 0.0       # CLIP contrastive label smoothing
# gradient accumulation simulates batch_size*GRAD_ACCUM_STEPS
# This means effective batch = 256*2 = 512 without extra VRAM
GRAD_ACCUM_STEPS = 2     # Accumulate gradients over N steps before optimizer.step()

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


Using device: cuda


## Data Ingestion

In [3]:
import os
import json

def read_coco_pairs(images_dir: str, token_file: str):
    """Parses MS-COCO JSON file → list of (image_path, caption) pairs."""
    with open(token_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # Map image IDs to file names
    id_to_filename = {img['id']: img['file_name'] for img in data['images']}
    available_images = set(os.listdir(images_dir))
    
    pairs = []
    for ann in data['annotations']:
        img_id = ann['image_id']
        caption = ann['caption']
        if img_id in id_to_filename:
            filename = id_to_filename[img_id]
            if filename in available_images:
                img_path = os.path.abspath(os.path.join(images_dir, filename))
                pairs.append((img_path, caption))
                
    return pairs

# ── Load MS-COCO (Train + Val splits) ─────────────────────────────────────────
# ── Load MS-COCO (Train + Val splits) ─────────────────────────────────────────
coco_train_dir  = r"/home/zahid/Desktop/NanoVLM/data/train2017"
coco_train_json = r"/home/zahid/Desktop/NanoVLM/data/annotations_trainval2017/annotations/captions_train2017.json"

coco_val_dir    = r"/home/zahid/Desktop/NanoVLM/data/val2017"
coco_val_json   = r"/home/zahid/Desktop/NanoVLM/data/annotations_trainval2017/annotations/captions_val2017.json"


print("Loading COCO train pairs...")
train_pairs = read_coco_pairs(coco_train_dir, coco_train_json)
print(f"Train pairs: {len(train_pairs)}")

print("Loading COCO val pairs...")
val_pairs = read_coco_pairs(coco_val_dir, coco_val_json)
print(f"Val pairs: {len(val_pairs)}")

all_texts = [c for _, c in train_pairs]


Loading COCO train pairs...
Train pairs: 591753
Loading COCO val pairs...
Val pairs: 25014


## Tokenizer (Simple Word-Level)

In [4]:
class SimpleTokenizer:
    def __init__(self, min_freq=2, max_vocab_size=10000):
        self.min_freq = min_freq
        self.max_vocab_size = max_vocab_size
        self.pad_token, self.unk_token = "<pad>", "<unk>"
        self.bos_token, self.eos_token = "<bos>", "<eos>"
        self.specials = [self.pad_token, self.unk_token, self.bos_token, self.eos_token]
        self.stoi = {tok: i for i, tok in enumerate(self.specials)}
        self.itos = list(self.specials)

    def _tokenize(self, text: str):
        return re.findall(r"\w+|[^\w\s]", text.lower())

    @property
    def pad_id(self): return self.stoi[self.pad_token]
    @property
    def unk_id(self): return self.stoi[self.unk_token]
    @property
    def bos_id(self): return self.stoi[self.bos_token]
    @property
    def eos_id(self): return self.stoi[self.eos_token]

    def build_vocab(self, texts: List[str]):
        counter = Counter()
        for t in texts:
            counter.update(self._tokenize(t))
        max_new = self.max_vocab_size - len(self.specials)
        sorted_tokens = sorted(
            [(tok, freq) for tok, freq in counter.items() if freq >= self.min_freq],
            key=lambda x: (-x[1], x[0])
        )[:max_new]
        for tok, _ in sorted_tokens:
            if tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)

    def encode(self, text: str, max_len=64):
        tokens = self._tokenize(text)
        ids = [self.bos_id] + [self.stoi.get(t, self.unk_id) for t in tokens] + [self.eos_id]
        return torch.tensor(ids[:max_len], dtype=torch.long)

    def decode(self, ids: torch.Tensor):
        if ids.ndim > 1:
            return [self.decode(seq) for seq in ids]
        tokens = [self.itos[i] for i in ids.tolist()
                  if i not in [self.pad_id, self.bos_id, self.eos_id]]
        return " ".join(tokens)


## Image Processing & Dataset

In [5]:
class ResizeLongestSide:
    """Resize so the longest side = max_side (downscale only)."""
    def __init__(self, max_side=224):
        self.max_side = max_side

    def __call__(self, img: Image.Image):
        w, h = img.size
        scale = self.max_side / max(w, h)
        if scale < 1.0:
            nw, nh = int(round(w * scale)), int(round(h * scale))
            img = VF.resize(img, (nh, nw), interpolation=transforms.InterpolationMode.BICUBIC)
        return img


class PairedImageTextDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, tokenizer, img_transform=None, max_text_len=64):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.img_transform = img_transform
        self.max_text_len = max_text_len

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img_path, text = self.pairs[idx]
        img = Image.open(img_path).convert("RGB")
        if self.img_transform:
            img = self.img_transform(img)
        toks = self.tokenizer.encode(text, max_len=self.max_text_len)
        return img, toks, text


def vlm_collate_fn(batch, pad_token_id=0):
    """Pads images and token sequences to the same size within a batch."""
    imgs, toks, texts = zip(*batch)

    # Pad images to the same H×W
    max_h = max(x.shape[1] for x in imgs)
    max_w = max(x.shape[2] for x in imgs)
    padded_imgs = []
    for x in imgs:
        pad = (0, max_w - x.shape[2], 0, max_h - x.shape[1])
        padded_imgs.append(F.pad(x, pad, value=0.0))
    batch_imgs = torch.stack(padded_imgs)

    # Pad token sequences and build attention mask
    max_l = max(t.shape[0] for t in toks)
    batch_toks = torch.full((len(toks), max_l), pad_token_id, dtype=torch.long)
    batch_mask = torch.zeros((len(toks), max_l), dtype=torch.bool)
    for i, t in enumerate(toks):
        batch_toks[i, :t.shape[0]] = t
        batch_mask[i, :t.shape[0]] = True

    return batch_imgs, batch_toks, batch_mask, list(texts)


## Build Dataset & DataLoaders

In [6]:
# ── No need to split manually, COCO is already split! ─────────────────────
# We already have train_pairs and val_pairs from the previous cell.

# Build vocabulary from TRAINING captions only (never look at val)
all_texts = [c for _, c in train_pairs]
tokenizer = SimpleTokenizer(min_freq=2, max_vocab_size=10000)
tokenizer.build_vocab(all_texts)
print(f"Vocab size: {len(tokenizer.itos)}")

# ── Separate train/val transforms ─────────────────────────────────────────
_NORM_MEAN = [0.48145466, 0.4578275,  0.40821073]   # CLIP ImageNet stats
_NORM_STD  = [0.26862954, 0.26130258, 0.27577711]

img_tf_train = transforms.Compose([
    ResizeLongestSide(max_side=256),
    transforms.RandomResizedCrop(224,
                                 scale=(0.65, 1.0),
                                 interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.3, contrast=0.3,
        saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=_NORM_MEAN, std=_NORM_STD),
])

img_tf_val = transforms.Compose([
    ResizeLongestSide(max_side=224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=_NORM_MEAN, std=_NORM_STD),
])

train_dataset = PairedImageTextDataset(train_pairs, tokenizer, img_transform=img_tf_train, max_text_len=64)
val_dataset   = PairedImageTextDataset(val_pairs,   tokenizer, img_transform=img_tf_val,   max_text_len=64)

collate = lambda b: vlm_collate_fn(b, pad_token_id=tokenizer.pad_id)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True, collate_fn=collate)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True, collate_fn=collate)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


Vocab size: 10000
Train batches: 2312 | Val batches: 98


## Image Encoder (CNN + Projection)

In [7]:
from torchvision import models as tv_models

class ImageEncoder(nn.Module):
    """
    Vision encoder: MobileNetV2 backbone → spatial token projection →
    self-attention over image regions → mean pooling → MLP head.

    MobileNetV2 outputs a 7×7 spatial grid (for 224×224 input), giving 49
    spatial tokens.  Self-attention lets the model learn which regions of the
    image carry semantic weight for a given caption.
    """
    def __init__(self, embd_dim=EMBD_DIM, dropout=DROPOUT):
        super().__init__()

        # Pretrained backbone with partial freezing
        backbone = tv_models.mobilenet_v2(weights=tv_models.MobileNet_V2_Weights.DEFAULT).features

        # Freeze first 14 blocks
        for i, child in enumerate(backbone.children()):
            if i < 14:
                for p in child.parameters():
                    p.requires_grad = False

        self.backbone   = backbone
        self.dropout    = nn.Dropout(p=dropout)
        self.projection = nn.Linear(1280, embd_dim)      # per-token projection

        # Pre-norm self-attention
        self.attn_norm = nn.LayerNorm(embd_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=embd_dim, num_heads=ATTEN_HEADS, batch_first=True, dropout=dropout
        )

        # Pre-norm MLP
        self.mlp_norm = nn.LayerNorm(embd_dim)
        self.mlp_head = nn.Sequential(
            nn.Linear(embd_dim, embd_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embd_dim * 2, embd_dim),
        )

        self.final_norm = nn.LayerNorm(embd_dim)

    def forward(self, x):
        x = self.backbone(x)                              # [B, 1280, 7, 7]
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)                  # [B, 49, 1280]
        x = self.dropout(self.projection(x))               # [B, 49, embd_dim]

        # Self-attention
        x_norm = self.attn_norm(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out                                  # [B, 49, embd_dim]

        # Per-token MLP
        x = x + self.mlp_head(self.mlp_norm(x))           # [B, 49, embd_dim]

        # Pool
        x = x.mean(dim=1)                                 # [B, embd_dim]

        return F.normalize(self.final_norm(x), dim=-1)


## Text Encoder (Transformer)

In [8]:

class TextEncoder(nn.Module):
    """
    Embedding → N-layer Transformer → mean pool → MLP head → L2-normalised embedding.

    """
    def __init__(self, embd_dim=EMBD_DIM, num_heads=ATTEN_HEADS,
                 vocab_size=10000, context_window=64,
                 num_layers=NUM_TX_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, embd_dim)
        self.position_embedding = nn.Embedding(context_window, embd_dim)

        # 6-layer transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embd_dim, nhead=num_heads,
            dim_feedforward=embd_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embd_dim)

        # MLP projection head (similar to CLIP's text projection)
        # This gives the model extra capacity to map language representations
        # into the shared image-text embedding space.
        self.mlp_head = nn.Sequential(
            nn.Linear(embd_dim, embd_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embd_dim * 2, embd_dim),
        )

    def forward(self, toks, attention_mask=None):
        N, L = toks.shape
        pos = torch.arange(L, device=toks.device).unsqueeze(0)
        x   = self.token_embedding(toks) + self.position_embedding(pos)

        pad_mask = (attention_mask == 0) if attention_mask is not None else None
        x = self.transformer(x, src_key_padding_mask=pad_mask)   # [B, L, embd_dim]

        # Mean-pool over valid (non-padding) tokens
        x = x.mean(dim=1)                                         # [B, embd_dim]

        # MLP head with residual connection
        x = x + self.mlp_head(x)

        return F.normalize(self.norm(x), dim=-1)                  # unit-norm embedding


## CLIP Contrastive Loss

In [9]:

class CLIPLoss(nn.Module):
    """
    Symmetric cross-entropy over the image-text similarity matrix.

    """
    def __init__(self, label_smoothing=0.1):
        super().__init__()
        # Learnable log-scale temperature (real CLIP initialisation)
        self.logit_scale    = nn.Parameter(torch.ones([]) * (1 / 0.07))
        # store label smoothing factor
        self.label_smoothing = label_smoothing

    def forward(self, image_emb, txt_emb):
        scale   = self.logit_scale.exp().clamp(max=100)
        logits  = image_emb @ txt_emb.T * scale      # [B, B] similarity matrix
        targets = torch.arange(image_emb.size(0), device=image_emb.device)

        # label_smoothing=0.1 on both directions
        loss_i = F.cross_entropy(logits,   targets, label_smoothing=self.label_smoothing)
        loss_t = F.cross_entropy(logits.T, targets, label_smoothing=self.label_smoothing)
        return (loss_i + loss_t) / 2.0


## Model, Optimizer & Scheduler

In [10]:
img_enc   = ImageEncoder().to(device)
txt_enc   = TextEncoder(vocab_size=len(tokenizer.itos)).to(device)

# Safety defaults for out-of-order notebook execution
if 'LR' not in globals():
    LR = 3e-4
if 'WEIGHT_DECAY' not in globals():
    WEIGHT_DECAY = 1e-4
if 'WARMUP_EPOCHS' not in globals():
    WARMUP_EPOCHS = 3
if 'EARLY_STOP_PATIENCE' not in globals():
    EARLY_STOP_PATIENCE = 8
if 'LABEL_SMOOTH' not in globals():
    LABEL_SMOOTH = 0.0

criterion = CLIPLoss(label_smoothing=LABEL_SMOOTH).to(device)   # learnable temperature lives here

# All parameters: both encoders + the learnable temperature
params    = list(img_enc.parameters()) + list(txt_enc.parameters()) + list(criterion.parameters())

# AdamW with cosine-decayed LR and weight decay for regularisation
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

# Linear warmup + cosine decay (standard for CLIP).
# Warmup gradually ramps LR from ~0 → LR over WARMUP_EPOCHS, then cosine-decays.
# This prevents large noisy updates from destabilising the model at the start.
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)   # linear ramp
    # cosine decay from LR → ~0 over remaining epochs
    progress = (epoch - WARMUP_EPOCHS) / max(1, epochs - WARMUP_EPOCHS)
    return max(1e-6 / LR, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Mixed-precision scaler — trains in float16 for speed + lower VRAM
scaler = GradScaler('cuda')

# ── Resume from checkpoint if one exists ────────────────────────────────────
import os
start_epoch   = 0
best_val_loss_init = float('inf')

CKPT_FILES = {
    'img_enc'  : 'best_img_enc.pth',
    'txt_enc'  : 'best_txt_enc.pth',
    'criterion': 'best_criterion.pth',
    'optimizer': 'best_optimizer.pth',
    'scheduler': 'best_scheduler.pth',
    'meta'     : 'best_meta.pth',
}

if all(os.path.exists(v) for v in list(CKPT_FILES.values())[:3]):   # weights must exist
    print("\n✓ Checkpoint found — resuming training...")
    img_enc.load_state_dict(torch.load(CKPT_FILES['img_enc'],   map_location=device))
    txt_enc.load_state_dict(torch.load(CKPT_FILES['txt_enc'],   map_location=device))
    criterion.load_state_dict(torch.load(CKPT_FILES['criterion'], map_location=device))
    if os.path.exists(CKPT_FILES['optimizer']):
        optimizer.load_state_dict(torch.load(CKPT_FILES['optimizer'], map_location=device))
    if os.path.exists(CKPT_FILES['scheduler']):
        scheduler.load_state_dict(torch.load(CKPT_FILES['scheduler'], map_location=device))
    if os.path.exists(CKPT_FILES['meta']):
        meta = torch.load(CKPT_FILES['meta'], map_location='cpu')
        start_epoch        = meta.get('epoch', 0) + 1
        best_val_loss_init = meta.get('best_val_loss', float('inf'))
    print(f"  Resuming from epoch {start_epoch + 1}/{epochs}  |  best val loss so far: {best_val_loss_init:.4f}")
else:
    print("No checkpoint found — starting from epoch 1.")

print(f"ImageEncoder params : {sum(p.numel() for p in img_enc.parameters()):,}")
print(f"TextEncoder params  : {sum(p.numel() for p in txt_enc.parameters()):,}")
print(f"Temperature (init)  : {criterion.logit_scale.item():.4f}")

# EMA: shadow weights updated as: ema = decay * ema + (1-decay) * param
EMA_DECAY = 0.999

ema_img = AveragedModel(
    img_enc,
    avg_fn=lambda ema, new, n: EMA_DECAY * ema + (1 - EMA_DECAY) * new
)
ema_txt = AveragedModel(
    txt_enc,
    avg_fn=lambda ema, new, n: EMA_DECAY * ema + (1 - EMA_DECAY) * new
)
print(f"EMA decay           : {EMA_DECAY}")

# ── TensorBoard Writer ────────────────────────────────────────────────────────
tb_writer = SummaryWriter(log_dir="runs/nanoVLM")
print(f"TensorBoard logs    : runs/nanoVLM")



✓ Checkpoint found — resuming training...
  Resuming from epoch 22/50  |  best val loss so far: 3.2314
ImageEncoder params : 3,079,424
TextEncoder params  : 7,578,368
Temperature (init)  : 3.1110
EMA decay           : 0.999
TensorBoard logs    : runs/nanoVLM


## Pre-Training Embedding Snapshot

In [11]:
# Record embeddings BEFORE training to compare with post-training
img_enc.eval(); txt_enc.eval()
with torch.no_grad():
    sample_idx = random.randrange(len(val_dataset))
    sample_img, sample_toks, sample_caption = val_dataset[sample_idx]
    s_img  = sample_img.unsqueeze(0).to(device)
    s_toks = sample_toks.unsqueeze(0).to(device)
    s_mask = (s_toks != tokenizer.pad_id).to(device)

    pre_img_emb = img_enc(s_img).squeeze(0).cpu().numpy()
    pre_txt_emb = txt_enc(s_toks, attention_mask=s_mask).squeeze(0).cpu().numpy()

cos_sim_pre = float(
    np.dot(pre_img_emb, pre_txt_emb) /
    (np.linalg.norm(pre_img_emb) * np.linalg.norm(pre_txt_emb) + 1e-12)
)
print(f"Pre-training cosine similarity: {cos_sim_pre:.4f}")
print(f"Img emb shape: {pre_img_emb.shape} | Txt emb shape: {pre_txt_emb.shape}")


Pre-training cosine similarity: 0.2053
Img emb shape: (256,) | Txt emb shape: (256,)


## Training Loop

In [ ]:
# Safety fallback — in case checkpoint cell was skipped
if 'start_epoch' not in dir():
    start_epoch = 0
if 'best_val_loss_init' not in dir():
    best_val_loss_init = float('inf')
if 'CKPT_FILES' not in dir():
    CKPT_FILES = {
        'img_enc'  : 'best_img_enc.pth',
        'txt_enc'  : 'best_txt_enc.pth',
        'criterion': 'best_criterion.pth',
        'optimizer': 'best_optimizer.pth',
        'scheduler': 'best_scheduler.pth',
        'meta'     : 'best_meta.pth',
    }

print(f"Starting training on {device} for {epochs} epochs...", flush=True)
print(f"Epochs to run: {start_epoch + 1} → {epochs}", flush=True)
print(f"Hyperparams: dropout={DROPOUT} | weight_decay={WEIGHT_DECAY} | "
      f"label_smooth={LABEL_SMOOTH} | warmup={WARMUP_EPOCHS}ep | patience={EARLY_STOP_PATIENCE}ep", flush=True)

best_val_loss    = best_val_loss_init
train_losses, val_losses = [], []   # track for plotting
no_improve_count = 0                # early-stopping counter

for epoch in range(start_epoch, epochs):
    # ── Training ─────────────────────────────────────────────────────────────
    img_enc.train(); txt_enc.train(); criterion.train()
    total_loss = 0.0

    for batch_idx, (batch_imgs, batch_toks, batch_mask, _) in enumerate(train_loader):
        batch_imgs = batch_imgs.to(device, non_blocking=True)
        batch_toks = batch_toks.to(device, non_blocking=True)
        batch_mask = batch_mask.to(device, non_blocking=True)

        # gradient accumulation — zero_grad only at start of each accumulation window.
        # Effective batch size = batch_size × GRAD_ACCUM_STEPS (more negatives, stronger signal).
        if batch_idx % GRAD_ACCUM_STEPS == 0:
            optimizer.zero_grad()

        with autocast('cuda'):
            img_emb = img_enc(batch_imgs)
            txt_emb = txt_enc(batch_toks, attention_mask=batch_mask)
            # Divide loss by GRAD_ACCUM_STEPS so accumulated gradients are correctly scaled.
            loss    = criterion(img_emb, txt_emb) / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        # only step optimizer after accumulating GRAD_ACCUM_STEPS gradients
        # (or at the very last batch of the epoch).
        is_last_batch = (batch_idx + 1) == len(train_loader)
        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or is_last_batch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

        # Clamp logit_scale in log-space after each optimiser step (OpenAI practice)
        with torch.no_grad():
            criterion.logit_scale.clamp_(max=np.log(100))

        # ── EMA update ───────────────────────────────────────────────────────
        # After each optimizer step, update the shadow EMA copies.
        # These copies are used ONLY for evaluation — they produce smoother
        # embeddings because they average over many training steps.
        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or is_last_batch:
            ema_img.update_parameters(img_enc)
            ema_txt.update_parameters(txt_enc)

        # Multiply back by GRAD_ACCUM_STEPS to log the true unscaled loss value.
        total_loss += loss.item() * GRAD_ACCUM_STEPS

        if batch_idx % 20 == 0:
            true_loss = loss.item() * GRAD_ACCUM_STEPS
            print(f"  Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{len(train_loader)} "
                  f"| Loss: {true_loss:.4f} | Temp: {criterion.logit_scale.exp().item():.2f}",
                  flush=True)

        # ── TensorBoard: log every batch ─────────────────────────────────────
        # global_step gives TensorBoard a unique x-axis value across all epochs.
        global_step = epoch * len(train_loader) + batch_idx
        tb_writer.add_scalar("batch/train_loss", loss.item() * GRAD_ACCUM_STEPS, global_step)
        tb_writer.add_scalar("batch/temperature", criterion.logit_scale.exp().item(), global_step)

    avg_train_loss = total_loss / len(train_loader)

    # step the LR scheduler once per epoch (warmup + cosine)
    scheduler.step()

    # ── Validation ───────────────────────────────────────────────────────────
    img_enc.eval(); txt_enc.eval(); criterion.eval()
    val_loss = 0.0
    with torch.no_grad():
        for b_imgs, b_toks, b_mask, _ in val_loader:
            b_imgs = b_imgs.to(device, non_blocking=True)
            b_toks = b_toks.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)
            with autocast('cuda'):
                val_loss += criterion(img_enc(b_imgs), txt_enc(b_toks, attention_mask=b_mask)).item()

    avg_val_loss = val_loss / len(val_loader)
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    current_lr = scheduler.get_last_lr()[0]
    print(f"\n--- Epoch {epoch+1}/{epochs} | Train: {avg_train_loss:.4f} "
          f"| Val: {avg_val_loss:.4f} | LR: {current_lr:.2e} | Temp: {criterion.logit_scale.exp().item():.2f}", flush=True)

    # ── TensorBoard: log epoch-level metrics ─────────────────────────────────
    # These appear as smooth epoch-level curves in TensorBoard.
    tb_writer.add_scalar("epoch/train_loss", avg_train_loss, epoch + 1)
    tb_writer.add_scalar("epoch/val_loss",   avg_val_loss,   epoch + 1)
    tb_writer.add_scalar("epoch/learning_rate", current_lr,  epoch + 1)
    tb_writer.add_scalar("epoch/temperature", criterion.logit_scale.exp().item(), epoch + 1)

    # ── Checkpoint (best val loss) ────────────────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        no_improve_count = 0
        torch.save(img_enc.state_dict(),   CKPT_FILES['img_enc'])
        torch.save(txt_enc.state_dict(),   CKPT_FILES['txt_enc'])
        torch.save(criterion.state_dict(), CKPT_FILES['criterion'])
        torch.save(optimizer.state_dict(), CKPT_FILES['optimizer'])
        torch.save(scheduler.state_dict(), CKPT_FILES['scheduler'])
        torch.save({'epoch': epoch, 'best_val_loss': best_val_loss}, CKPT_FILES['meta'])
        print(f"  ✓ Checkpoint saved (epoch {epoch+1}, val_loss={best_val_loss:.4f}).", flush=True)
    else:
        no_improve_count += 1
        print(f"  ✗ No improvement ({no_improve_count}/{EARLY_STOP_PATIENCE})", flush=True)

    # early stopping
    if no_improve_count >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1} — no val improvement for {EARLY_STOP_PATIENCE} epochs.", flush=True)
        break

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

# ── Close TensorBoard writer ─────────────────────────────────────────────────
# Flush any buffered data and release the log file handle.
tb_writer.close()
print("TensorBoard logs saved to runs/nanoVLM")


## Training Curves

In [22]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss")
plt.plot(range(1, len(val_losses)+1),   val_losses,   label="Val Loss")
plt.xlabel("Epoch"); plt.ylabel("CLIP Loss")
plt.title("Training vs Validation Loss")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.show()


NameError: name 'train_losses' is not defined

<Figure size 900x400 with 0 Axes>

## Post-Training Evaluation

In [23]:
# Load best checkpoint
img_enc.load_state_dict(torch.load("best_img_enc.pth", map_location=device))
txt_enc.load_state_dict(torch.load("best_txt_enc.pth", map_location=device))
img_enc.eval(); txt_enc.eval()

# Compute cosine similarity on same sample as pre-training snapshot
with torch.no_grad():
    post_img_emb = img_enc(s_img).squeeze(0).cpu().numpy()
    post_txt_emb = txt_enc(s_toks, attention_mask=s_mask).squeeze(0).cpu().numpy()

cos_sim_post = float(
    np.dot(post_img_emb, post_txt_emb) /
    (np.linalg.norm(post_img_emb) * np.linalg.norm(post_txt_emb) + 1e-12)
)
improvement = cos_sim_post - cos_sim_pre

print(f"Pre-training  cosine similarity : {cos_sim_pre:.4f}")
print(f"Post-training cosine similarity : {cos_sim_post:.4f}")
print(f"Improvement                     : {improvement:+.4f}")

if improvement <= 0:
    print("⚠ No improvement detected on this sample.")
    print("  Try lower learning rate, larger batch (or grad accumulation), and earlier stopping.")


Pre-training  cosine similarity : 0.2909
Post-training cosine similarity : 0.2909
Improvement                     : +0.0000
⚠ No improvement detected on this sample.
  Try lower learning rate, larger batch (or grad accumulation), and earlier stopping.


In [24]:
import torch
import os

# Check the metadata from your best run
if os.path.exists("best_meta.pth"):
    meta = torch.load("best_meta.pth")
    print(f"✅ Metadata found!")
    print(f"Best Epoch: {meta.get('epoch')}")
    print(f"Best Val Loss recorded: {meta.get('val_loss'):.4f}")
else:
    print("❌ best_meta.pth not found!")

# Check if the weights actually load without error now
try:
    img_enc.load_state_dict(torch.load("best_img_enc.pth", map_location=device))
    print("✅ Image weights loaded successfully!")
except Exception as e:
    print(f"❌ Image loading error: {e}")


✅ Metadata found!
Best Epoch: 20


TypeError: unsupported format string passed to NoneType.__format__

In [ ]:
import torch

# 1. Initialize models
img_enc = ImageEncoder().to(device)
txt_enc = TextEncoder().to(device)

# 2. Load the best weights
img_enc.load_state_dict(torch.load("best_img_enc.pth", map_location=device))
txt_enc.load_state_dict(torch.load("best_txt_enc.pth", map_location=device))
img_enc.eval(); txt_enc.eval()

# 3. Initialize EMA and sync them (mandatory for the evaluation cell)
from torch.optim.swa_utils import AveragedModel
ema_img = AveragedModel(img_enc)
ema_txt = AveragedModel(txt_enc)

print("✅ Models initialized and COCO weights loaded!")


In [ ]:
import torch

# Load the vocabulary from the checkpoint if it exists
if os.path.exists("best_meta.pth"):
    meta = torch.load("best_meta.pth")
    if 'tokenizer_state' in meta:
        tokenizer.load_state(meta['tokenizer_state'])
        print("✅ Tokenizer synchronized with training vocab!")
    else:
        print("⚠️ Tokenizer state not in meta. Building from scratch (RISKY)...")
        # If it's not in meta, make sure you build it from the EXACT same train_pairs
        tokenizer.build_vocab([c for _, c in train_pairs])
else:
    # If no meta, we must hope building from train_pairs is consistent
    tokenizer.build_vocab([c for _, c in train_pairs])

print(f"Current Vocab Size: {len(tokenizer.itos)}")


## Top-K Retrieval Accuracy

In [12]:
# Faster post-training evaluation (prevents machine hangs on large val sets)
# - Caps number of evaluated batches by default
# - Uses one top-k pass per batch (faster than repeating topk for each K)
# - Skips live-vs-EMA double pass unless explicitly enabled

def evaluate_recall(img_encoder, txt_encoder, loss_fn, loader, ks=(1, 5, 10), max_batches=30):
    """
    Compute batch-local Recall@K and average loss.

    max_batches:
        None -> evaluate full loader
        int  -> evaluate first N batches only (recommended for quick checks)
    """
    img_encoder.eval()
    txt_encoder.eval()

    ks = tuple(sorted(set(ks)))
    max_k = max(ks)

    total_samples = 0
    total_loss = 0.0
    total_batches = 0
    correct_at_k = {k: 0 for k in ks}

    use_amp = device.type == "cuda"

    with torch.inference_mode():
        for batch_idx, (b_imgs, b_toks, b_mask, _) in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            b_imgs = b_imgs.to(device, non_blocking=True)
            b_toks = b_toks.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            with autocast(device_type="cuda", enabled=use_amp):
                img_emb = img_encoder(b_imgs)
                txt_emb = txt_encoder(b_toks, attention_mask=b_mask)
                batch_loss = loss_fn(img_emb, txt_emb)

            total_loss += batch_loss.item()
            total_batches += 1

            scores = img_emb @ txt_emb.T
            B = scores.size(0)
            total_samples += B
            targets = torch.arange(B, device=scores.device).unsqueeze(1)

            top_k_indices = torch.topk(scores, k=min(max_k, B), dim=1).indices

            for k in ks:
                if k > B:
                    correct_at_k[k] += B
                    continue
                hits = (top_k_indices[:, :k] == targets).any(dim=1)
                correct_at_k[k] += hits.sum().item()

    recalls = {k: (correct_at_k[k] / max(total_samples, 1)) for k in ks}
    avg_loss = total_loss / max(total_batches, 1)
    return recalls, avg_loss, total_batches


# Load best checkpoint into live models
img_enc.load_state_dict(torch.load("best_img_enc.pth", map_location=device))
txt_enc.load_state_dict(torch.load("best_txt_enc.pth", map_location=device))

# Sync EMA weights to loaded best checkpoint
ema_img.update_parameters(img_enc)
ema_txt.update_parameters(txt_enc)

EVAL_MAX_BATCHES = 30      # set to None for full validation set
RUN_LIVE_COMPARISON = False  # set True only when you need a second full pass

if EVAL_MAX_BATCHES is None:
    print("Evaluating EMA model on FULL validation set...\n")
else:
    print(f"Evaluating EMA model on first {EVAL_MAX_BATCHES} validation batches...\n")

recalls, ema_val_loss, used_batches = evaluate_recall(
    ema_img.module, ema_txt.module, criterion, val_loader, ks=(1, 5, 10), max_batches=EVAL_MAX_BATCHES
)

print(f"EMA Val Loss: {ema_val_loss:.4f}  (batches used: {used_batches})")
for k, r in recalls.items():
    print(f"  Image->Text Recall@{k:>2d}: {r:.1%}")

if RUN_LIVE_COMPARISON:
    print("\n--- Comparison: Live model (no EMA) ---")
    recalls_live, live_val_loss, _ = evaluate_recall(
        img_enc, txt_enc, criterion, val_loader, ks=(1, 5, 10), max_batches=EVAL_MAX_BATCHES
    )
    print(f"Live Val Loss: {live_val_loss:.4f}")
    for k, r in recalls_live.items():
        diff = recalls[k] - r
        print(f"  Image->Text Recall@{k:>2d}: {r:.1%}  (EMA {'+' if diff >= 0 else ''}{diff:.1%})")
else:
    print("\nSkipped live-vs-EMA comparison to keep evaluation lightweight.")


Evaluating EMA model on first 30 validation batches...

EMA Val Loss: 2.6436  (batches used: 30)
  Image->Text Recall@ 1: 19.2%
  Image->Text Recall@ 5: 68.7%
  Image->Text Recall@10: 84.8%

Skipped live-vs-EMA comparison to keep evaluation lightweight.
